In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.manifold import TSNE

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Transformations
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Load CIFAR-10
train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

Using device: cuda


In [2]:
# 1. LeNet-5 Implementation
class LeNet5(nn.Module):
    def __init__(self):
        super(LeNet5, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 6, 5), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(6, 16, 5), nn.ReLU(), nn.MaxPool2d(2, 2)
        )
        self.classifier = nn.Sequential(
            nn.Linear(16 * 5 * 5, 120), nn.ReLU(),
            nn.Linear(120, 84), nn.ReLU(),
            nn.Linear(84, 10)
        )
    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)

# 2. AlexNet (CIFAR-10 Version)
class AlexNet(nn.Module):
    def __init__(self):
        super(AlexNet, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, 3, stride=1, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 192, 3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(192, 384, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(384, 256, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, 3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
        )
        self.classifier = nn.Sequential(
            nn.Dropout(), nn.Linear(256 * 4 * 4, 4096), nn.ReLU(inplace=True),
            nn.Dropout(), nn.Linear(4096, 4096), nn.ReLU(inplace=True),
            nn.Linear(4096, 10)
        )
    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)

# 3. VGGNet (VGG16 Architecture)
class VGGNet(nn.Module):
    def __init__(self):
        super(VGGNet, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1), nn.ReLU(), nn.Conv2d(64, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.Conv2d(128, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(128, 256, 3, padding=1), nn.ReLU(), nn.Conv2d(256, 256, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(256, 512, 3, padding=1), nn.ReLU(), nn.Conv2d(512, 512, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2, 2)
        )
        self.classifier = nn.Linear(512 * 2 * 2, 10)
    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)

# 4. ResNet (Simplified ResNet-18/50 style blocks)
class BasicBlock(nn.Module):
    def __init__(self, in_planes, planes, stride=1):
        super(BasicBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, planes, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(planes)
            )
    def forward(self, x):
        out = torch.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        return torch.relu(out)

class ResNet(nn.Module):
    def __init__(self, block, num_blocks):
        super(ResNet, self).__init__()
        self.in_planes = 64
        self.conv1 = nn.Conv2d(3, 64, 3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.layer1 = self._make_layer(block, 64, num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, 128, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, 256, num_blocks[2], stride=2)
        self.layer4 = self._make_layer(block, 512, num_blocks[3], stride=2)
        self.linear = nn.Linear(512, 10)

    def _make_layer(self, block, planes, num_blocks, stride):
        strides = [stride] + [1]*(num_blocks-1)
        layers = []
        for s in strides:
            layers.append(block(self.in_planes, planes, s))
            self.in_planes = planes
        return nn.Sequential(*layers)

    def forward(self, x):
        out = torch.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        out = torch.nn.functional.avg_pool2d(out, 4)
        out = out.view(out.size(0), -1)
        return out, self.linear(out)

In [3]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce_loss = nn.CrossEntropyLoss()(inputs, targets)
        pt = torch.exp(-ce_loss)
        return self.alpha * (1 - pt)**self.gamma * ce_loss

class ArcFaceLoss(nn.Module):
    def __init__(self, in_features, out_features, s=30.0, m=0.50):
        super(ArcFaceLoss, self).__init__()
        self.weight = nn.Parameter(torch.FloatTensor(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)
        self.s, self.m = s, m

    def forward(self, input, label):
        cosine = nn.functional.linear(nn.functional.normalize(input), nn.functional.normalize(self.weight))
        sine = torch.sqrt(1.0 - torch.pow(cosine, 2))
        phi = cosine * np.cos(self.m) - sine * np.sin(self.m)
        one_hot = torch.zeros(cosine.size(), device=device)
        one_hot.scatter_(1, label.view(-1, 1).long(), 1)
        output = (one_hot * phi) + ((1.0 - one_hot) * cosine)
        return nn.CrossEntropyLoss()(output * self.s, label)

In [4]:
experiments = [
    {"Model": "VGGNet", "Optimizer": "Adam", "Epochs": 10, "Loss": "BCE"},
    {"Model": "AlexNet", "Optimizer": "SGD", "Epochs": 20, "Loss": "Focal Loss"},
    {"Model": "ResNet", "Optimizer": "Adam", "Epochs": 15, "Loss": "ArcFace"}
]

results = []

for exp in experiments:
    print(f"\n--- Training {exp['Model']} with {exp['Loss']} ---")
    
    # Model Selection
    if exp['Model'] == "VGGNet": model = VGGNet().to(device)
    elif exp['Model'] == "AlexNet": model = AlexNet().to(device)
    else: model = ResNet(BasicBlock, [2, 2, 2, 2]).to(device) # ResNet-18 for speed
    
    # Optimizer Selection
    if exp['Optimizer'] == "Adam": optimizer = optim.Adam(model.parameters(), lr=0.001)
    else: optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
    
    # Loss Selection
    if exp['Loss'] == "BCE": criterion = nn.CrossEntropyLoss()
    elif exp['Loss'] == "Focal Loss": criterion = FocalLoss()
    else: 
        # ArcFace uses features as input
        feat_dim = 512 if exp['Model'] == "ResNet" else 4096
        criterion = ArcFaceLoss(feat_dim, 10).to(device)

    # Training
    for epoch in range(exp['Epochs']):
        model.train()
        train_correct, train_total = 0, 0
        for imgs, lbls in train_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            optimizer.zero_grad()
            
            if exp['Model'] == "ResNet": features, outputs = model(imgs)
            else: 
                outputs = model(imgs)
                features = outputs # Use logits as features for non-resnet simplified
            
            loss = criterion(features, lbls) if exp['Loss'] == "ArcFace" else criterion(outputs, lbls)
            loss.backward()
            optimizer.step()
            
            _, pred = outputs.max(1)
            train_total += lbls.size(0)
            train_correct += pred.eq(lbls).sum().item()
            
    # Evaluation
    model.eval()
    test_correct, test_total = 0, 0
    with torch.no_grad():
        for imgs, lbls in test_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            if exp['Model'] == "ResNet": _, outputs = model(imgs)
            else: outputs = model(imgs)
            _, pred = outputs.max(1)
            test_total += lbls.size(0)
            test_correct += pred.eq(lbls).sum().item()

    exp_result = {
        "Model": exp['Model'],
        "Optimizer": exp['Optimizer'],
        "Epochs": exp['Epochs'],
        "Loss Function": exp['Loss'],
        "Training Accuracy": f"{100.*train_correct/train_total:.2f}%",
        "Testing Accuracy": f"{100.*test_correct/test_total:.2f}%"
    }
    results.append(exp_result)
    print(f"Finished: {exp_result['Testing Accuracy']} test accuracy.")

# Create and Save Table
df = pd.DataFrame(results)
df.to_csv("lab_practical_results.csv", index=False)
print("\nFinal Table:\n", df)


--- Training VGGNet with BCE ---
Finished: 78.09% test accuracy.

--- Training AlexNet with Focal Loss ---
Finished: 81.92% test accuracy.

--- Training ResNet with ArcFace ---
Finished: 4.70% test accuracy.

Final Table:
      Model Optimizer  Epochs Loss Function Training Accuracy Testing Accuracy
0   VGGNet      Adam      10           BCE            92.91%           78.09%
1  AlexNet       SGD      20    Focal Loss            89.81%           81.92%
2   ResNet      Adam      15       ArcFace             3.69%            4.70%
